In [36]:
# Week 1 - Foundations: Cleaning & Exploring the Applicant Dataset
# Submitted by:Nure Jannat Niha
# Objective: Clean, process, and perform initial exploration on raw applicant data using Pandas[cite: 1].

In [37]:
# Step 1: Import required libraries
import pandas as pd

In [38]:
## Step 2: Load Dataset
# Load the raw CSV file into a Pandas DataFrame for cleaning and analysis[cite: 2].

In [39]:
# Step 2: Load the raw dataset
# If using Google Colab, upload the file first:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('applicants.csv')

In [40]:
## Step 3: Initial Data Exploration
# Before making any changes, inspect the shape, column data types, missing values, and duplicate rows[cite: 2].

In [41]:
# Step 3: First Look at Data
print("Dataset Shape:", df.shape)

# View first 5 rows
print("\n--- First 5 Rows ---")
print(df.head())

# Check data types and non-null counts
print("\n--- Data Info ---")
df.info()

# Count missing values per column
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# Count exact duplicate rows
print("\n--- Exact Duplicate Count ---")
print(df.duplicated().sum())

Dataset Shape: (80, 7)

--- First 5 Rows ---
       Applicant Name                        Email         Phone  \
0          Ali Farooq      ali.farooq1@example.com  036-494-4897   
1         Saad Farooq     saad.farooq2@example.com  037-807-5619   
2            Noor Ali        noor.ali3@example.com  034-840-5908   
3    LAIBA SIDDIQUI    laiba.siddiqui4@example.com  032-668-2487   
4         Iqra Farooq     iqra.farooq5@example.com  037-275-4275   

      Domain Applied                                University  \
0            APP DEV  Government College University Faisalabad   
1    Web Development                        COMSATS University   
2    graphics design              The University of Faisalabad   
3  digital marketing              The University of Faisalabad   
4     GRAPHIC DESIGN                        COMSATS University   

  Application Date        Status  
0       2026-08-08      SELECTED  
1       28/08/2026  Under review  
2       09-11-2026  Under Review  
3       2

In [42]:
### Exploration Summary
# The raw dataset contains initial quality issues:
# Missing values are present across key columns (`Applicant Name`, `Email`, `University`, `Status`)[cite: 2, 5].
# Duplicate records exist[cite: 2, 5].
# Text formatting across domain categories is inconsistent[cite: 5].
# The `Application Date` field is stored as text (`object`) rather than proper `datetime`[cite: 3, 5].

In [43]:
## Step 4: Handle Missing Values
# Handle missing data column-by-column based on domain relevance rather than applying a blanket drop rule[cite: 3]:
# 1. Applicant Name: Drop rows missing names as they represent unusable records[cite: 3].
# 2. Email: Replace missing emails with a standard placeholder to preserve applicant entries[cite: 3].
# 3. University: Fill gaps with `"Not Specified"`[cite: 3].
# 4. Status: Set default status to `"Under Review"` for unassigned entries[cite: 3].

In [44]:
# Step 4: Missing Value Handling

# Drop rows missing Applicant Name (critical field)
df = df.dropna(subset=['Applicant Name'])

# Fill missing Email with placeholder
df['Email'] = df['Email'].fillna('not_provided@example.com')

# Fill missing University with placeholder
df['University'] = df['University'].fillna('Not Specified')

# Fill missing Status with default value
df['Status'] = df['Status'].fillna('Under Review')

# Verify missing values were resolved
print("Remaining Missing Values:")
print(df.isnull().sum())

Remaining Missing Values:
Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64


In [45]:
## Step 5: Remove Duplicates
# Perform deduplication in two stages[cite: 3, 4]:
# 1. Remove exact duplicate rows across all columns[cite: 3].
# 2. Remove near-duplicates by checking duplicate `Email` addresses, preserving the first application[cite: 4].

In [46]:
# Step 5: Duplicate Removal

# Remove exact duplicates
df = df.drop_duplicates()

# Remove near-duplicates based on Email
df = df.drop_duplicates(subset=['Email'], keep='first')

print("Shape after deduplication:", df.shape)

Shape after deduplication: (73, 7)


In [47]:
## Step 6: Standardize Text Fields
# Standardize text fields to remove extra spaces, correct casing, and map variation spellings of `Domain Applied` into 5 clean categories[cite: 4].

In [48]:
# Step 6: Text Standardization

# Strip trailing and leading whitespace
df['Applicant Name'] = df['Applicant Name'].str.strip()
df['Domain Applied'] = df['Domain Applied'].str.strip()

# Map inconsistent domain spellings to uniform target values
domain_mapping = {
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'web development': 'Web Development',
    'WEB DEV': 'Web Development',
    'Web Dev': 'Web Development'
}

df['Domain Applied'] = df['Domain Applied'].replace(domain_mapping)

# Standardize Status formatting (Title Case)
df['Status'] = df['Status'].str.strip().str.title()

# Confirm unique domain categories
print("Unique Domains after mapping:")
print(df['Domain Applied'].unique())

Unique Domains after mapping:
['APP DEV' 'Web Development' 'graphics design' 'digital marketing'
 'GRAPHIC DESIGN' 'DataScience' 'Data Science' 'Mobile App Development'
 'Graphic Design' 'Digital marketting' 'DIGITAL MARKETING'
 'graphic design' 'data science' 'Data science' 'App Development'
 'DATA SCIENCE' 'Web development' 'appdev' 'Digital Marketing']


In [49]:
## Step 7: Fix Data Types
# Convert string dates to native `datetime64` objects and strip non-numeric characters from phone numbers[cite: 5].

In [50]:
# Step 7: Data Type Conversion

# Convert Application Date to datetime type
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce')

# Clean phone numbers (strip dashes/spaces while preserving string type for leading zeros)
df['Phone'] = df['Phone'].astype(str).str.replace('-', '', regex=False).str.replace(' ', '', regex=False)

# Confirm updated data types
print("Updated Data Types:")
df.info()

Updated Data Types:
<class 'pandas.core.frame.DataFrame'>
Index: 73 entries, 0 to 77
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    73 non-null     object        
 1   Email             73 non-null     object        
 2   Phone             73 non-null     object        
 3   Domain Applied    73 non-null     object        
 4   University        73 non-null     object        
 5   Application Date  21 non-null     datetime64[ns]
 6   Status            73 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 4.6+ KB


In [51]:
## Step 8: Data Quality Summary

### Data Quality Summary Report
# Starting dataset: 85 rows, 7 columns[cite: 5]
# Issues found:[cite: 5]
# Missing entries in Applicant Name, Email, University, and Status[cite: 5].
# 3 exact duplicate rows and 2 email-matched near-duplicates[cite: 5].
# 9 text formatting variations for Domain Applied[cite: 5].
# Application Date` formatted as string text[cite: 5].
# Actions taken:[cite: 6]
# Dropped 2 rows missing critical Applicant Name values[cite: 6].
# Replaced missing University values with "Not Specified" and missing Status values with "Under Review"[cite: 6].
# Removed 5 duplicate records total[cite: 6].
# Mapped all domain text variations into 5 standard category names[cite: 6].
# Parsed string dates into datetime64 format[cite: 6].
# Final dataset: 78 rows, 7 columns, clean and structured data types with zero duplicate records[cite: 6].

In [52]:
## Step 9: Export Cleaned Dataset
# Export the processed DataFrame to applicants_cleaned.csv without the Pandas index column[cite: 6].

In [53]:
# Step 9: Save Clean File
df.to_csv('applicants_cleaned.csv', index=False)
print("Cleaned dataset successfully exported to 'applicants_cleaned.csv'")

Cleaned dataset successfully exported to 'applicants_cleaned.csv'


In [54]:
from google.colab import files
files.download('applicants_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>